In [1]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from transformers import BertTokenizer, BertModel, BertForMaskedLM, BertForQuestionAnswering
import matplotlib.pyplot as plt

import torchvision.models as models
from torchvision import transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

e:\Code\seniorProject\aiModels\languageModel\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [10]:
#Load values from pretrained model
textCheckpoint = torch.load("../languageModel/combined_model.pth", map_location=device)
emotionsList = ["sadness", "joy", "love", "anger", "fear", "surprise"]
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

bert = BertModel.from_pretrained("bert-base-uncased")
bert.to(device)

classifier = nn.Sequential(
    nn.Linear(bert.config.hidden_size, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, len(emotionsList)))
classifier.to(device)

bert.load_state_dict(textCheckpoint["bert_state_dict"])
classifier.load_state_dict(textCheckpoint["classifier_state_dict"])

bert.to(device)
classifier.to(device)

Sequential(
  (0): Linear(in_features=768, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=256, out_features=6, bias=True)
)

In [31]:
def predict_emotion(text):
    # Set to evaluation mode
    bert.eval()
    classifier.eval()

    # Disable gradient calculation for faster inference
    with torch.no_grad():
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=64)

        # Move inputs to device
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        outputs = bert(input_ids, attention_mask)
        logits = classifier(outputs.pooler_output)

        pred = torch.argmax(logits, dim=1).item()
        prob = torch.softmax(logits, dim=1)

    return emotionsList[pred], prob.squeeze().cpu().numpy()

In [32]:
predict_emotion("I am so disheartened")

('sadness',
 array([9.9996603e-01, 1.9052513e-07, 2.2002600e-13, 9.3701618e-07,
        3.2878783e-05, 6.0428442e-14], dtype=float32))

In [ ]:
num_classes = 7
IMAGE_EMOTION_LABELS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

class FERResNet(nn.Module):
    """
    ResNet18 backbone with a custom emotion classification head.
    Dropout added for regularization on the small FER dataset.
    """
    def __init__(self, num_classes=7, dropout=0.4, freeze_until_layer=None):
        super().__init__()
        # Pretrained ResNet18 backbone
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        # freeze early layers (useful for fine-tuning on CK+)
        if freeze_until_layer:
            layers_to_freeze = ['conv1', 'bn1', 'layer1', 'layer2']
            for name, param in base.named_parameters():
                if any(name.startswith(l) for l in layers_to_freeze[:freeze_until_layer]):
                    param.requires_grad = False

        in_features = base.fc.in_features
        base.fc = nn.Identity()        # Remove original classifier
        self.backbone = base

        # Custom emotion head
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

    def get_trainable_params(self):
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')



model = FERResNet(num_classes=num_classes).to(device)
ckpt  = torch.load("../facialModel/fer_ck_finetuned_inferese.pth", map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

FERResNet(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_

In [3]:
val_transform = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def predict_face(image_path):

    model.eval()

    image = Image.open(image_path)
    image = val_transform(image)
    image = image.unsqueeze(0).to(device)  # add batch dimension

    outputs = model(image)

    probs = torch.softmax(outputs, dim=1)
    predicted_class = torch.argmax(probs, dim=1).item()

    print(probs)

    return IMAGE_EMOTION_LABELS[predicted_class], probs.squeeze().cpu().numpy()

predict_face('./testImage.jpg')

tensor([[0.0993, 0.1871, 0.1847, 0.0561, 0.3541, 0.0534, 0.0653]],
       device='cuda:0')


('Sad',
 array([0.09929284, 0.18711342, 0.1847113 , 0.05614145, 0.35406983,
        0.05339328, 0.06527784], dtype=float32))